# Aegis Ocean: Maritime Vessel Detection with YOLOv8-OBB

**Author:** Md Zubayer Patowari | **Date:** February 2026

### **Project Overview**
This notebook provides a comprehensive, end-to-end workflow for training a state-of-the-art **YOLOv8-OBB (Oriented Bounding Box)** model for ship detection in Synthetic Aperture Radar (SAR) imagery. It is designed for maximum reproducibility and can be executed seamlessly in Google Colab (for GPU acceleration) or a local Jupyter environment.

### **Technical Objectives**
1.  **Environment Setup:** Install and verify all necessary libraries.
2.  **Data Connection & Configuration:** Prepare the SAR dataset for training by connecting to the data source and configuring paths.
3.  **Model Training:** Fine-tune a pre-trained YOLOv8-OBB model on the custom SAR ship dataset.
4.  **Performance Evaluation:** Quantitatively assess the model's accuracy using standard object detection metrics (mAP, Precision, Recall).
5.  **Results Visualization:** Qualitatively analyze the model's performance by visualizing training graphs and predictions on test images.

---
## Step 1: Environment Setup

This section handles dependency installation and environment configuration. If running locally, ensure you have a Python 3.9+ environment.

In [ ]:
# Install Ultralytics YOLOv8 (Quiet mode)
%pip install ultralytics -q

# Import necessary libraries
import os
import sys
import yaml
from ultralytics import YOLO
import torch
import ultralytics

# Verify installation and check for GPU
print('Setup Complete.')
print(f'PyTorch Version: {torch.__version__}')
print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'CPU\'}')
ultralytics.checks()

---
## Step 2: Connect to Data

This cell automatically detects if you are running in **Google Colab** or a **Local Environment** and configures the dataset path accordingly.

*   **Colab Users:** Ensure you have uploaded the SSDD_OBB folder to your Google Drive.
*   **Local Users:** Ensure the SSDD_OBB folder is in the project directory.

In [ ]:
# --- Environment Auto-Detection & Data Connection ---

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DATA_ROOT = None

if IN_COLAB:
    print(' Detected Google Colab environment.')
    print('   > Mounting Google Drive...')
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Define the root directory in Drive
    # IMPORTANT: Ensure 'SSDD_OBB' is uploaded to the root of your Drive or update this path.
    DATA_ROOT = '/content/drive/MyDrive/SSDD_OBB'
    
    if not os.path.exists(DATA_ROOT):
        print(f'\\n WARNING: Dataset folder not found at: {DATA_ROOT}')
        print('   Please upload the \'SSDD_OBB\' folder to your Google Drive.')
    else:
        print(f'\\n Dataset found at: {DATA_ROOT}')

else:
    print(' Detected Local environment.')
    
    # Auto-detect local path
    possible_paths = [
        os.path.abspath('SSDD_OBB'),       # Current directory
        os.path.join(os.getcwd(), 'SSDD_OBB'),
        os.path.abspath(os.path.join(os.getcwd(), '..', 'SSDD_OBB'))
    ]
    
    for path in possible_paths:
        if os.path.exists(path):
            DATA_ROOT = path
            break
            
    if DATA_ROOT:
         print(f' Local dataset found at: {DATA_ROOT}')
    else:
        print(f'\\n Error: Dataset not found.')
        print(f'   Checked locations: {possible_paths}')
        print('   Please ensure \'SSDD_OBB\' folder exists locally.')

# Final Check
if DATA_ROOT:
    print(f'   > Active Data Root: {DATA_ROOT}')

---
## Step 3: Configure Dataset Path

The data_obb.yaml file tells YOLOv8 where to find the training, validation, and test data. We dynamically update this file to point to the DATA_ROOT detected above.

In [ ]:
if DATA_ROOT is None:
    print(' Process Stopped: No valid DATA_ROOT found. Please fix Step 2.')
else:
    # Define the path to the YAML configuration file
    yaml_path = os.path.join(DATA_ROOT, 'data.yaml')
    # Check if data_obb.yaml exists, otherwise check data.yaml
    if not os.path.exists(yaml_path):
        yaml_path = os.path.join(DATA_ROOT, 'data_obb.yaml')

    # Read and Update the YAML file
    try:
        with open(yaml_path, 'r') as f:
            data_config = yaml.safe_load(f)

        # Update the 'path' key to the correct root directory
        data_config['path'] = DATA_ROOT

        # Save the updated configuration back to the file
        with open(yaml_path, 'w') as f:
            yaml.dump(data_config, f)

        print(f' Successfully updated configuration.')
        print(f'   Config File: {yaml_path}')
        print(f'   Dataset Path set to: {DATA_ROOT}')

    except FileNotFoundError:
        print(f' Error: Could not find configuration file at {yaml_path}')

---
## Step 4: Model Training

We will train the **YOLOv8-Small-OBB (yolov8s-obb.pt)** model. This variant offers a strong balance between speed and accuracy for satellite imagery.

*   **Epochs:** 50 (Recommended for research baseline).
*   **Image Size:** 640 pixels.
*   **Batch Size:** 16 (Adjust based on GPU memory).

In [ ]:
# Load a pretrained YOLOv8-OBB model
model = YOLO('yolov8s-obb.pt')

# Determine the device to use (GPU or CPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f' Starting training on device: {device}')

if device == 'cpu':
    print(' WARNING: Training on CPU will be slow. Consider using Google Colab for GPU.')
    epochs = 2 # Reduced for CPU demonstration
else:
    epochs = 50

if DATA_ROOT:
    # Train the model
    results = model.train(
        data=yaml_path,
        epochs=epochs,
        imgsz=640,
        batch=16,
        name='aegis_ocean_model',
        device=device,
        exist_ok=True
    )
else:
    print(' Cannot train: Data root not defined.')

---
## Step 5: Evaluate Model Performance

Validating on the test split ensures unbiased performance metrics (mAP, Precision, Recall).

In [ ]:
if 'results' in locals():
    # Run validation on the 'test' split using the best trained weights
    run_dir = results.save_dir
    best_model_path = os.path.join(run_dir, 'weights', 'best.pt')
    
    if os.path.exists(best_model_path):
        print(f'Loading best model from: {best_model_path}')
        val_model = YOLO(best_model_path)
        metrics = val_model.val(split='test')
        
        print('\\n --- Final Test Set Metrics ---')
        print(f'mAP50-95 (Primary Metric): {metrics.box.map:.4f}')
        print(f'mAP50:    {metrics.box.map50:.4f}')
        print(f'Precision:{metrics.box.p[0] if len(metrics.box.p) > 0 else 0:.4f}')
        print(f'Recall:   {metrics.box.r[0] if len(metrics.box.r) > 0 else 0:.4f}')
    else:
        print('Best model weights not found. Did training complete?')
else:
    print('No training results variable found.')

---
## Step 6: Visualize Results

Visual analysis of predictions and learning curves.

In [ ]:
from IPython.display import Image, display
import glob

if 'run_dir' in locals():
    # 1. Training Curves
    results_png = os.path.join(run_dir, 'results.png')
    if os.path.exists(results_png):
        print(' Training & Validation Curves:')
        display(Image(filename=results_png))
    
    # 2. Confusion Matrix
    conf_matrix = os.path.join(run_dir, 'confusion_matrix.png')
    if os.path.exists(conf_matrix):
        print('\\nmatrix Confusion Matrix:')
        display(Image(filename=conf_matrix))
    
    # 3. Sample Detections
    print('\\n --- Sample Test Detections ---')
    if DATA_ROOT:
        test_imgs = glob.glob(os.path.join(DATA_ROOT, 'test', 'images', '*.jpg'))[:3]
        
        if test_imgs:
            # Run inference
            val_model.predict(source=test_imgs, save=True, project='predictions', name='test_samples', exist_ok=True)
            
            # Show results
            for img in test_imgs:
                pred_path = os.path.join('predictions', 'test_samples', os.path.basename(img))
                if os.path.exists(pred_path):
                    display(Image(filename=pred_path))
        else:
            print('No test images found to visualize within DATA_ROOT.')
    else:
        print('DATA_ROOT not found.')
else:
    print('Run directory not defined.')